In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D2 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D2 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd



In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D2"
BRANCH_ID = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 22

EXPECTED_FIELDS = [
    "Line Item",
    "Unit",
    "Value 2024",
    "Value 2023"
]

REFERENCE_FIELDS = EXPECTED_FIELDS + ["Source Location"]

ALIGNMENT_IDENTITY_FIELDS = [
    "Line Item"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Unit",
    "Value 2024",
    "Value 2023"
]

NUMERIC_FIELDS = [
    "Value 2024",
    "Value 2023"
]

ALLOWED_UNITS = {
    "EUR millions",
    "EUR"
}

NUMERIC_TOLERANCE = 0.0

OUTPUT_DIR = Path("outputs_D2_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH_ID, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)

In [ ]:
# ============================================================
# 2. Upload of validation inputs
# ============================================================
# Required:
#   1) D2_reference_values.csv
#   2) D2_branch_B_parsed_extraction.json
#   3) D2_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    f for f in uploaded_files
    if f.lower().endswith(".csv")
]

json_files = [
    f for f in uploaded_files
    if f.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one Stage 1 reference-values CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]

PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(file_name, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if (
        isinstance(obj, dict)
        and isinstance(obj.get("records"), list)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        isinstance(obj, dict)
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH_ID
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D2 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D2 Branch B technical diagnostics."
    )

print("Reference values:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)

In [ ]:
# ============================================================
# 3. Load inputs and verify provenance
# ============================================================

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8"
) as f:
    extraction_json = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8"
) as f:
    technical_diagnostics = json.load(f)

df_ref = pd.read_csv(REFERENCE_FILE)
df_ext_raw = pd.DataFrame(
    extraction_json["records"]
)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "technical diagnostics": technical_diagnostics
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id."
        )

    if artefact.get("branch") != BRANCH_ID:
        raise ValueError(
            f"Unexpected {artefact_name} branch."
        )


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8192),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


input_provenance = {
    "reference_file":
        REFERENCE_FILE,

    "reference_sha256":
        sha256_file(REFERENCE_FILE),

    "parsed_extraction_file":
        PARSED_EXTRACTION_FILE,

    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,

    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE)
}

print("Reference shape:", df_ref.shape)
print("Extraction shape:", df_ext_raw.shape)

In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

schema_diagnostics = {
    "valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity
}

print("Imported Branch B technical/schema status:")
print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ------------------------------------------------------------
# 5. Verify Stage 1 reference and extraction fields
# ------------------------------------------------------------

missing_reference_fields = [
    field for field in REFERENCE_FIELDS
    if field not in df_ref.columns
]

if missing_reference_fields:
    raise ValueError(
        f"Stage 1 reference dataset is missing fields: {missing_reference_fields}"
    )

if len(df_ref) != EXPECTED_RECORD_COUNT:
    raise ValueError(
        f"Unexpected Stage 1 reference count: {len(df_ref)}"
    )

missing_extraction_columns = [
    field for field in EXPECTED_FIELDS
    if field not in df_ext_raw.columns
]

df_ext = df_ext_raw.copy()


for field in missing_extraction_columns:
    df_ext[field] = np.nan

df_ref = df_ref[REFERENCE_FIELDS].copy()
df_ext = df_ext[EXPECTED_FIELDS].copy()

print("Reference records:", len(df_ref))
print("Extracted records:", len(df_ext))
print("Missing extraction columns:", missing_extraction_columns)


In [ ]:
# ------------------------------------------------------------
# 6. Deterministic comparison normalisation
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))

    for char in [
        "\u00a0", "\u2000", "\u2001", "\u2002", "\u2003",
        "\u2004", "\u2005", "\u2006", "\u2007", "\u2008",
        "\u2009", "\u200a", "\u202f", "\u205f", "\u3000"
    ]:
        text = text.replace(char, " ")

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("–", "-")
        .replace("—", "-")
    )

    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def normalize_number(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()

    if text == "":
        return np.nan

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", "").replace(" ", "")

    if text.startswith("(") and text.endswith(")"):
        text = "-" + text[1:-1]

    text = text.replace(",", "")

    try:
        return float(text)
    except (TypeError, ValueError):
        return np.nan


def numbers_match(reference_value, extracted_value):
    ref_num = normalize_number(reference_value)
    ext_num = normalize_number(extracted_value)

    if pd.isna(ref_num) and pd.isna(ext_num):
        return True

    if pd.isna(ref_num) or pd.isna(ext_num):
        return False

    return math.isclose(
        ref_num,
        ext_num,
        rel_tol=0.0,
        abs_tol=NUMERIC_TOLERANCE
    )


In [ ]:
# ------------------------------------------------------------
# 7. Creation of D2 observation keys and check uniqueness
# ------------------------------------------------------------

df_ref["match_key"] = df_ref["Line Item"].apply(normalize_text)
df_ext["match_key"] = df_ext["Line Item"].apply(normalize_text)

reference_duplicate_count = int(
    df_ref["match_key"].duplicated(keep=False).sum()
)

if reference_duplicate_count > 0:
    raise ValueError(
        "The fixed Stage 1 reference contains duplicate normalised Line Item keys."
    )

extraction_duplicate_mask = df_ext["match_key"].duplicated(keep="first")
duplicate_extraction_records = df_ext[extraction_duplicate_mask].copy()
df_ext_unique = df_ext[~extraction_duplicate_mask].copy()

print("Reference duplicate observations:", reference_duplicate_count)
print("Additional extracted duplicate observations:", len(duplicate_extraction_records))


In [ ]:
# ------------------------------------------------------------
# 8. One-to-one record alignment
# ------------------------------------------------------------

df_validation = df_ref.merge(
    df_ext_unique,
    on="match_key",
    how="outer",
    suffixes=("_ref", "_ext"),
    indicator=True,
    validate="one_to_one"
)

print(df_validation["_merge"].value_counts(dropna=False))


In [ ]:
# ------------------------------------------------------------
# 9. Field-level comparison for aligned observations
# ------------------------------------------------------------

matched_mask = df_validation["_merge"] == "both"

# Line Item
df_validation["Line Item_match"] = False
df_validation.loc[matched_mask, "Line Item_match"] = df_validation.loc[
    matched_mask
].apply(
    lambda row:
        normalize_text(row["Line Item_ref"])
        == normalize_text(row["Line Item_ext"]),
    axis=1
)

# Unit
df_validation["Unit_match"] = False
df_validation.loc[matched_mask, "Unit_match"] = df_validation.loc[
    matched_mask
].apply(
    lambda row:
        normalize_text(row["Unit_ref"])
        == normalize_text(row["Unit_ext"]),
    axis=1
)

# Numerical values
for field in NUMERIC_FIELDS:
    col = f"{field}_match"
    df_validation[col] = False
    df_validation.loc[matched_mask, col] = df_validation.loc[
        matched_mask
    ].apply(
        lambda row: numbers_match(
            row[f"{field}_ref"],
            row[f"{field}_ext"]
        ),
        axis=1
    )

PRIMARY_MATCH_COLUMNS = [
    f"{field}_match"
    for field in PRIMARY_CORRECTNESS_FIELDS
]

df_validation["all_primary_fields_match"] = (
    matched_mask
    & df_validation[
        PRIMARY_MATCH_COLUMNS
    ].all(axis=1)
)


In [ ]:
# ------------------------------------------------------------
# 10. Classify record outcomes
# ------------------------------------------------------------

def classify_record(row):
    if row["_merge"] == "left_only":
        return "missing"
    if row["_merge"] == "right_only":
        return "hallucinated_unsupported"
    if bool(row["all_primary_fields_match"]):
        return "fully_correct"
    return "discrepant"

df_validation["record_status"] = df_validation.apply(classify_record, axis=1)

missing_records = df_validation[
    df_validation["record_status"] == "missing"
].copy()

unsupported_records = df_validation[
    df_validation["record_status"] == "hallucinated_unsupported"
].copy()

discrepant_records = df_validation[
    df_validation["record_status"] == "discrepant"
].copy()

fully_correct_records_df = df_validation[
    df_validation["record_status"] == "fully_correct"
].copy()

duplicate_extraction_records["record_status"] = "hallucinated_duplicate"

print("Missing:", len(missing_records))
print("Unsupported unmatched:", len(unsupported_records))
print("Unsupported duplicate extras:", len(duplicate_extraction_records))
print("Discrepant matched:", len(discrepant_records))
print("Fully correct:", len(fully_correct_records_df))


In [ ]:
# ------------------------------------------------------------
# 11. Calculate common validation metrics
# ------------------------------------------------------------

N_REF = int(len(df_ref))
N_EXT = int(len(df_ext))
N_ALIGNED = int((df_validation["_merge"] == "both").sum())
N_MISSING = int(len(missing_records))
N_UNSUPPORTED_UNMATCHED = int(len(unsupported_records))
N_DUPLICATE_EXTRAS = int(len(duplicate_extraction_records))
N_HALLUCINATED = N_UNSUPPORTED_UNMATCHED + N_DUPLICATE_EXTRAS
N_DISCREPANT = int(len(discrepant_records))
N_CORRECT = int(len(fully_correct_records_df))

completeness = N_ALIGNED / N_REF if N_REF else 0.0

record_precision = N_CORRECT / N_EXT if N_EXT else 0.0
record_recall = N_CORRECT / N_REF if N_REF else 0.0
record_f1 = (
    2 * record_precision * record_recall / (record_precision + record_recall)
    if (record_precision + record_recall) > 0 else 0.0
)

hallucination_rate = N_HALLUCINATED / N_EXT if N_EXT else 0.0
missing_rate = N_MISSING / N_REF if N_REF else 0.0
discrepancy_rate = N_DISCREPANT / N_ALIGNED if N_ALIGNED else 0.0

matched_validation = df_validation[df_validation["_merge"] == "both"].copy()

field_accuracy_among_aligned = {}
for field in EXPECTED_FIELDS:
    col = f"{field}_match"
    field_accuracy_among_aligned[field] = (
        float(matched_validation[col].mean())
        if len(matched_validation) else 0.0
    )

correct_field_instances = int(
    matched_validation[
        PRIMARY_MATCH_COLUMNS
    ].sum().sum()
)

expected_field_instances = int(
    N_REF * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_field_instances / expected_field_instances
    if expected_field_instances else 0.0
)


In [ ]:
# ------------------------------------------------------------
# 12. Field-level error summary
# ------------------------------------------------------------

field_error_summary = []

for field in EXPECTED_FIELDS:
    col = f"{field}_match"

    correct_aligned = int(matched_validation[col].sum())
    incorrect_aligned = int(len(matched_validation) - correct_aligned)

    field_error_summary.append({
        "field": field,
        "used_in_alignment_identity":
            field in ALIGNMENT_IDENTITY_FIELDS,

        "used_in_primary_correctness":
            field in PRIMARY_CORRECTNESS_FIELDS,
        "aligned_records_evaluated": int(len(matched_validation)),
        "correct_values_among_aligned": correct_aligned,
        "incorrect_values_among_aligned": incorrect_aligned,
        "accuracy_among_aligned": (
            round(correct_aligned / len(matched_validation), 4)
            if len(matched_validation) else 0.0
        ),
        "missing_expected_instances": N_MISSING,
        "overall_correct_instances": correct_aligned,
        "overall_expected_instances": N_REF,
        "field_accuracy": (
            round(correct_aligned / N_REF, 4)
            if N_REF else 0.0
        )
    })

field_error_summary_df = pd.DataFrame(field_error_summary)
field_error_summary_df


In [ ]:
# ------------------------------------------------------------
# 13. Build final validation summary and export artefacts
# ------------------------------------------------------------

summary = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH_ID,
    "branch_name": BRANCH_NAME,

    "reference_records": N_REF,
    "extracted_records": N_EXT,
    "aligned_records": N_ALIGNED,

    "fully_correct_records": N_CORRECT,
    "discrepant_records": N_DISCREPANT,
    "missing_records": N_MISSING,

    "hallucinated_records": N_HALLUCINATED,
    "hallucinated_unmatched_records": N_UNSUPPORTED_UNMATCHED,
    "hallucinated_duplicate_records": N_DUPLICATE_EXTRAS,

    "completeness": round(completeness, 4),
    "missing_rate": round(missing_rate, 4),

    "record_precision_exact": round(record_precision, 4),
    "record_recall_exact": round(record_recall, 4),
    "record_f1_exact": round(record_f1, 4),

    "hallucination_rate": round(hallucination_rate, 4),
    "discrepancy_rate_among_aligned": round(discrepancy_rate, 4),

    "field_accuracy":
        round(field_accuracy, 4),

    "field_accuracy_among_aligned": {
        k: round(v, 4)
        for k, v in field_accuracy_among_aligned.items()
    },

    "schema_validity": schema_validity,
    "schema_diagnostics": schema_diagnostics,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "structurally_evaluable":
        structurally_evaluable,

    "comparison_rules_frozen_from_branch_A": True,

    "comparison_rules": {
        "text": (
            "Unicode NFKC, Unicode-space harmonisation, "
            "apostrophe/dash standardisation, whitespace collapse "
            "and case folding"
        ),
        "numeric":
            "Exact numerical equality after deterministic parsing",
        "numeric_tolerance": NUMERIC_TOLERANCE,
        "source_location_used_for_matching": False
    },

    "normalisation_note": (
        "Deterministic normalisation was applied only to comparison "
        "copies; the preserved Branch B extraction was not modified."
    ),

    "input_provenance": input_provenance
}

print("Final validation summary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

df_validation.to_csv(
    OUTPUT_DIR / "D2_branch_B_validation_detailed.csv",
    index=False
)

missing_records.to_csv(
    OUTPUT_DIR / "D2_branch_B_missing_records.csv",
    index=False
)

unsupported_records.to_csv(
    OUTPUT_DIR / "D2_branch_B_hallucinated_unmatched_records.csv",
    index=False
)

duplicate_extraction_records.to_csv(
    OUTPUT_DIR / "D2_branch_B_hallucinated_duplicate_records.csv",
    index=False
)

discrepant_records.to_csv(
    OUTPUT_DIR / "D2_branch_B_discrepant_records.csv",
    index=False
)

fully_correct_records_df.to_csv(
    OUTPUT_DIR / "D2_branch_B_fully_correct_records.csv",
    index=False
)

field_error_summary_df.to_csv(
    OUTPUT_DIR / "D2_branch_B_field_error_summary.csv",
    index=False
)

with open(
    OUTPUT_DIR / "D2_branch_B_validation_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Validation artefacts saved.")

In [ ]:
# ------------------------------------------------------------
# 14. Download validation artefacts
# ------------------------------------------------------------

for output_file in sorted(OUTPUT_DIR.iterdir()):
    if output_file.is_file():
        files.download(output_file)
